In [ ]:
import torch
from Model.DLSSNet.model_zhengjiao import Net
from Model.DLSSNet.args_zjBCIsingletrain_val import data_config
from torch.backends import cudnn
from einops import rearrange
import os
import torch.nn.functional as F
from scipy import io

cudnn.benchmark = False
cudnn.deterministic = True
subid = 7
model = Net(num_channels=data_config.num_channel, 
                    len_window=data_config.len_window, 
                    d_model=data_config.d_model, 
                    frame_stride=data_config.frame_stride,
                    num_frame=data_config.num_frame,
                    num_head=data_config.num_head, 
                    encoder_num_layers=data_config.encoder_num_layers, 
                    low_p = data_config.low_p,
                    dropout=data_config.dropout, 
                    transformerparwiseforward_dimrat=data_config.transformerparwiseforward_dimrat, 
                    statenum=data_config.statenum,
                    num_class=data_config.num_class)

device = torch.device("cpu")


target_category = 2  # set the class (class activation mapping)


path = 'C:\\2023Experiment\\parametersensitive\\单被试训练测试_matt数据_d_60\\BCICom_2a\\Conv+TransFormer+GPoolingV2+ClassTransHead+decoder+正交约束\\ON2024-11-22At13-10-37\\ckpl\\Fold'+str(subid).zfill(2)

# path = 'C:/2023Experiment/TrainingForanalysis/基群测试跨session/sub'+str(subid).zfill(2)+'/ckpl/State_num'+str(state_num).zfill(2)
cpkl_path = os.path.join(path, 'Conv+TransFormer+GPoolingV2+ClassTransHead+decoder+正交约束_best_params.pkl')
net_dict = torch.load(cpkl_path,map_location=device)
net_dict = net_dict['net_state_dict']
model.load_state_dict(net_dict)


data_path = 'C:/2023Experiment/DATA/BCICIV_2a_mat'


file_t = io.loadmat(os.path.join(data_path, 'BCIC_S' + f'{subid:02d}' + '_T.mat'))
file_e = io.loadmat(os.path.join(data_path, 'BCIC_S' + f'{subid:02d}' + '_E.mat'))

x_t = torch.Tensor(file_t['x_train'])
y_t = torch.Tensor(file_t['y_train']).view(-1)
x_e = torch.Tensor(file_e['x_test'])
y_e = torch.Tensor(file_e['y_test']).view(-1)

x_all = torch.concatenate([x_t, x_e], dim = 0)
y_all = torch.concatenate([y_t, y_e], dim = 0)

s = y_all.argsort()
x_all = x_all[s]
y_all = y_all[s]

dev = torch.device('cpu')

x_all = x_all[:, :, 124:562].to(dev)
y_all = y_all.long().to(dev)
model.eval()
with torch.no_grad():
    all_att, out, dec_x, embeded_x, basis, Pooled_x, Encoded_x, scaler, scores = model(x_all)

probabilities = F.softmax(out, dim=1)

# 获取最大概率的索引，即预测的类别
predictions = torch.argmax(probabilities, dim=1)

# 判断预测是否正确
correct = (predictions == y_all)

# 过滤掉分类错误的样本
correct_Pooled_x = Pooled_x[correct]
# correct_Pooled_x = Pooled_x
correct_labels = y_all[correct]
# correct_labels = y_all
correct_EmbededX = embeded_x[correct]
correct_x_all = x_all[correct]
# correct_EmbededX = embeded_x
correct_Encoded_x = Encoded_x[correct]
correct_scaler = scaler[correct]
correct_scores = scores[correct]

correct_Pooled_x = correct_Pooled_x.numpy()
correct_EmbededX = correct_EmbededX.numpy()
correct_Encoded_x = correct_Encoded_x.numpy()
correct_scaler = correct_scaler.numpy()
correct_x_all = correct_x_all.numpy()
correct_scores = correct_scores.numpy()


# # 过滤掉分类错误的样本
# correct_Pooled_x = Pooled_x
# # correct_Pooled_x = Pooled_x
# correct_labels = y_all
# # correct_labels = y_all
# correct_EmbededX = embeded_x
# correct_x_all = x_all
# # correct_EmbededX = embeded_x
# correct_Encoded_x = Encoded_x
# correct_scaler = scaler
# correct_Pooled_x = correct_Pooled_x.numpy()
# correct_EmbededX = correct_EmbededX.numpy()
# correct_Encoded_x = correct_Encoded_x.numpy()
# correct_scaler = correct_scaler.numpy()
# correct_x_all = correct_x_all.numpy()

In [ ]:
import numpy as np
import scipy.io as sio
params = model.state_dict()
p_1 = params['Embedding.TaskPotentialConvBlock.1.weight'].numpy()
p_2 = params['Embedding.TemporalStateConvBlock.0.weight'].numpy()
p_3 = params['Embedding.project.weight'].numpy()
p_3 = np.squeeze(p_3)
p_2 = np.squeeze(p_2)
p_1 = np.squeeze(p_1)

m_p_2 = []
for i in range(p_3.shape[0]):
    n_p_3 = p_3[i]
    n_p_3tom_p_2 = np.zeros((p_2.shape[1], p_2.shape[2]))
    for j in range(p_3.shape[1]):
        n_p_3tom_p_2 = n_p_3tom_p_2+np.dot(n_p_3[j], p_2[j])
    m_p_2.append(n_p_3tom_p_2[np.newaxis, :, :])

m_p_2 = np.concatenate(m_p_2)

m_p_1 = []
for i in range(m_p_2.shape[0]):
    m_p_2tom_p_1 = np.dot(np.transpose(p_1), m_p_2[i])
    m_p_1.append(m_p_2tom_p_1[np.newaxis, :, :])

m_p_1 = np.concatenate(m_p_1)
# min_val = np.min(m_p_1)
# max_val = np.max(m_p_1)
# scaled_m_p_1 = (m_p_1 - min_val) / (max_val - min_val)
# scaled_m_p_1 = m_p_1

In [ ]:
import mne
import matplotlib.pyplot as plt

def plot_eeg_topomap(eeg_weight, eeg_info,chLa_index, pattern_n, pattern_t_n, savepath,vlim):
    im, cn = mne.viz.plot_topomap(eeg_weight,
                                  eeg_info,
                                  names=chLa_index,
                                  vlim=vlim,
                                  size=5,
                                  cmap='RdBu_r',
                                  show=False,
                                  contours=0, 
                                  extrapolate='local')
    # plt.colorbar(im)
    plt.savefig(savepath+'Center_EEGpattern'+str(pattern_n).zfill(2)+'t'+str(pattern_t_n).zfill(2)+'.png', format='png', dpi=300) 
    # plt.show()
# ch_names = ['Fz', 'E2','E3', 'E4', 'E5', 'E6', 'E7', 'C3', 'E9', 'Cz', 'E11', 'C4', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'Pz', 'E21', 'E22']
biosemi_montage = mne.channels.make_standard_montage('biosemi64')
print(biosemi_montage)
index = [37, 9, 10, 46, 45, 44, 13, 12, 11, 47, 48, 49, 50, 17, 18, 31, 55, 54, 19, 30, 56, 29]  # for bci competition iv 2a
biosemi_montage.ch_names = [biosemi_montage.ch_names[i] for i in index]
biosemi_montage.dig = [biosemi_montage.dig[i+3] for i in index]
info = mne.create_info(ch_names=biosemi_montage.ch_names, sfreq=128., ch_types='eeg')
info.set_montage(biosemi_montage)
chLa_index = biosemi_montage.ch_names

        

In [ ]:
savepath = "C:/同步文件夹/2024-paper/图/functionalpatterns/"
if not os.path.exists(savepath):
    os.makedirs(savepath)
for i in range(m_p_1.shape[0]):
    raw = mne.io.RawArray(m_p_1[i], info)
    raw.plot(scalings = 0.06, duration=0.125, n_channels=m_p_1.shape[1], show=True)
    plt.savefig(savepath + "pattern_" + str(i) + ".png", format='png', dpi=300)

In [ ]:
savepath = "C:/同步文件夹/2024-paper/图/EEG_pattern_sub7d60_all_spatialonly/"
if not os.path.exists(savepath):
    os.makedirs(savepath)
# scaled_m_p_1 = np.tanh()
for i in range(m_p_1.shape[0]):
    for j in range(m_p_1.shape[2]):
        plot_eeg_topomap(m_p_1[i, :, j], info, chLa_index, i, j, savepath, vlim=(-np.max(m_p_1), np.max(m_p_1)))


In [ ]:
import scipy.io as sio
cluster_results = sio.loadmat('C:\\2023Experiment\\CodesRepository\\1task_cluster_centers605_sub07.mat')
task_cluster_labels = cluster_results['task_cluster_labels'][0]
sorted_by_distance = cluster_results['sorted_by_distance'][0]
cluster_center = cluster_results['task_cluster_centers'][sorted_by_distance]
# print(cluster_center)
# for i in range(len(cluster_center)):
    
# featuremap = np.matmul(cluster_center, p_1)

In [ ]:
import numpy as np
center_EEG = np.zeros((len(cluster_center), 22, 16))
for i in range(1, len(cluster_center)):
    center = cluster_center[i]
    center_pattern = np.zeros((22, 16))
    for j in range(len(center)):
        center_pattern = center_pattern + np.dot(center[j],m_p_1[j, :, :])
    center_EEG[i] = center_pattern
reref_center_EEG = center_EEG

# reref_center_EEG = np.zeros((len(cluster_center), 22, 16))
# for i in range(len(center_EEG)):
#     mean = np.mean(center_EEG[i], axis=0)
#     reref_center_EEG[i]= center_EEG[i]- mean

In [ ]:

energy_center_EEG = np.zeros((len(cluster_center), 22))
for i in range(len(cluster_center)):
    energy_center_EEG[i] = np.sum(reref_center_EEG[i], axis = 1)

In [ ]:
savepath = "C:/同步文件夹/2024-paper/图/EEG_pattern_sub7d60_clusters_all_energy/"
if not os.path.exists(savepath):
    os.makedirs(savepath)
for i in range(energy_center_EEG.shape[0]):
    plot_eeg_topomap(energy_center_EEG[i], info, chLa_index, i, 0, savepath, vlim=(0, np.max(energy_center_EEG[i])))

In [ ]:
tasks = {"left": 0, "right": 1, "foot": 2, "tongue": 3}
task_EmbeddedX = {}
task_PooledX = {}
task_scaler = {}
task_scores = {}
for task, id in tasks.items():
    index = np.where(correct_labels==tasks[task])
    task_EmbeddedX[task] = correct_EmbededX[index]
    task_PooledX[task] = correct_Pooled_x[index]
    task_scaler[task] = correct_scaler[index]
    task_scores[task] = correct_scores[index]

In [ ]:
task_PooledX['left'].shape

In [ ]:
savepath = "C:/同步文件夹/2024-paper/图/EEG_pattern_sub7d60_samples_stdmax/"
if not os.path.exists(savepath):
    os.makedirs(savepath)

tasks = [['left', 'right', 'foot', 'tongue'], ['left', 'right'],['foot', 'tongue'],['left', 'right', 'foot', 'tongue']]

for i in range(1, len(sorted_by_distance)):
    task_samples_EEG = {}
    all_data_forplot = []
    for l in range(len(tasks[i-1])):
        task = tasks[i-1][l]
        for j in range(task_PooledX[task].shape[0]):
            state_index = np.where(task_cluster_labels[task][0][j] == sorted_by_distance[i])[0]
            states = task_PooledX[task][j, state_index, :]
            all_data_forplot.append(states)
        task_samples_states = np.concatenate(all_data_forplot, axis = 0)
        task_samples_state = np.sum(task_samples_states, axis=0)
        task_samples_state = (task_samples_state-np.min(task_samples_state))/(np.max(task_samples_state)-np.min(task_samples_state))
        task_samples_pattern= np.zeros((22, 16))
        for k in range(len(task_samples_state)):
            task_samples_pattern = task_samples_pattern+ np.dot(task_samples_state[k],m_p_1[k, :, :])
        task_samples_EEG[task] = task_samples_pattern
        task_samples_pattern_top_std = np.std(task_samples_pattern, axis=0)
        max_index = np.argmax(task_samples_pattern_top_std)
        mean_task_samples_pattern = np.mean(task_samples_pattern[:, max_index])
        task_samples_pattern_top = task_samples_pattern[:, max_index]-mean_task_samples_pattern

        plot_eeg_topomap(task_samples_pattern_top, info, chLa_index, i, l, savepath, vlim=(np.min(task_samples_pattern_top), np.max(task_samples_pattern_top)))


In [ ]:
center_EEG_top = np.zeros((len(cluster_center), 22))
for i in range(len(cluster_center)):
    reref_center_EEG_top_std = np.std(reref_center_EEG[i], axis=0)
    max_index = np.argmax(reref_center_EEG_top_std)
    # mean_reref = np.mean(reref_center_EEG[i,:, max_index])
    center_EEG_top[i] = reref_center_EEG[i,:, max_index]


In [ ]:
U_all = []
s_all = []
VT_all = []
srat_all = []
for i in range(reref_center_EEG.shape[0]):
    U, s, VT = np.linalg.svd(reref_center_EEG[i], full_matrices=False)
    U_all.append(U)
    s_all.append(s)
    VT_all.append(VT)
    srat_all.append(s/np.sum(s))
    # 打印结果
    print("U矩阵:")
    print(U)
    print("\n奇异值:")
    print(s)
    print("\nV的转置矩阵:")
    print(VT)


In [ ]:
savepath = "C:/同步文件夹/2024-paper/图/EEG_pattern_sub7d60_clusters_svd/"
if not os.path.exists(savepath):
    os.makedirs(savepath)

K = 6
clusters_for_show_alll = np.zeros((len(U_all), K, 22))
for i in range(1, len(U_all)):
    for k in range(K):
        cluster_for_show = U_all[i][:, k]
        clusters_for_show_alll[i, k, :] = cluster_for_show

for i in range(len(U_all)):
    for k in range(K):
        plot_eeg_topomap(clusters_for_show_alll[i, k], info, chLa_index, i, k, savepath, vlim=(-np.max(clusters_for_show_alll), np.max(clusters_for_show_alll)))


In [ ]:
print(np.max(clusters_for_show_alll))

In [ ]:
savepath = "C:/同步文件夹/2024-paper/图/EEG_pattern_sub7d60_clusters_maxstdtop/"
if not os.path.exists(savepath):
    os.makedirs(savepath)
for i in range(center_EEG_top.shape[0]):
    plot_eeg_topomap(center_EEG_top[i], info, chLa_index, i, 0, savepath, vlim=(-1, 1))

In [ ]:
import mne
print(mne.__version__)

In [ ]:
import mne
raw_clusters = []
for i in range(len(cluster_center)):
    raw_cluster = mne.io.RawArray(reref_center_EEG[i], info)
    raw_clusters.append(raw_cluster)
    raw_cluster.plot(n_channels=22,scalings = 'auto', title='Cluster'+str(i).zfill(2), show=True, block=True)

In [ ]:
print(net_dict.keys())

In [ ]:
import numpy as np
center_EEG = np.zeros((len(cluster_center), 22, 16))
for i in range(len(cluster_center)):
    center = cluster_center[i]
    center_pattern = np.zeros((22, 16))
    for j in range(len(center)):
        center_pattern = center_pattern + np.dot(center[j],m_p_1[j, :, :])
    center_EEG[i] = center_pattern
# for i in range(16):
#     center_EEG[:,:,i] = center_EEG[:, :, i]- np.mean(center_EEG, axis=-1)


In [ ]:
scaled_center_EEG = np.zeros((len(cluster_center), 22, 16))
mean_scaled_center_EEG = np.zeros((len(cluster_center), 22))
savepath_mean =  "C:/同步文件夹/2024-paper/图/center_EEG_pattern_sub7_mean_d604/"
if not os.path.exists(savepath_mean):
    os.makedirs(savepath_mean)
for i in range(center_EEG.shape[0]):
    min_val = np.min(center_EEG[i])
    max_val = np.max(center_EEG[i])
    scaled_center_EEG = center_EEG
    scaled_center_EEG[i] = (center_EEG[i] - min_val) / (max_val - min_val)
    # scaled_center_EEG[i] = (scaled_center_EEG[i]- np.mean(scaled_center_EEG[i]))/np.std(scaled_center_EEG[i])
    for j in range(center_EEG.shape[2]):
        data= scaled_center_EEG[i, :, j]
        plot_eeg_topomap(data, info,chLa_index,i,j, savepath_mean)
    # mean_scaled_center_EEG[i] = np.mean(scaled_center_EEG[i], axis = -1)
    # plot_eeg_topomap(mean_scaled_center_EEG[i], info, chLa_index, i, i, savepath_mean)
    # for j in range(center_EEG.shape[2]):
    #     data= scaled_center_EEG[i, :, j]
    #     plot_eeg_topomap(data, info,chLa_index,i,j, savepath)

In [ ]:
# from scipy.stats import entropy
# center_EEG_st = np.std(center_EEG, axis=-1)
center_EEG_energy = np.zeros((center_EEG.shape[0],center_EEG.shape[1]))
for i in range(center_EEG.shape[0]):
    for j in range(center_EEG.shape[1]):
        center_EEG_energy[i, j] = np.sum(np.square(center_EEG[i, j, :]))

        # probability_distribution = np.bincount(center_EEG[i, j, :]) / len(center_EEG[i, j, :])
        # center_EEG_energy[i, j] = entropy(probability_distribution, base=2)
for i in range(center_EEG.shape[0]):
    center_EEG_energy[i, :] = center_EEG_energy[i, :]- np.mean(center_EEG_energy)
center_EEG_energy = 1 / (1 + np.exp(-center_EEG_energy))
savepath_energy =  "C:/同步文件夹/2024-paper/图/center_EEG_pattern_sub7_energyd60/"
if not os.path.exists(savepath_energy):
    os.makedirs(savepath_energy)
# scaled_center_EEG = np.zeros((len(cluster_center), 22))
# min_val = np.min(center_EEG_energy)
# max_val = np.max(center_EEG_energy)
# scaled_center_EEG = (center_EEG_energy - min_val) / (max_val - min_val)
for i in range(center_EEG_energy.shape[0]):
    # min_val = np.min(center_EEG_energy[i])
    # max_val = np.max(center_EEG_energy[i])
    # scaled_center_EEG[i] = (center_EEG_energy[i] - min_val) / (max_val - min_val)
    # scaled_center_EEG = scaled_center_EEG
    # for j in range(center_EEG.shape[2]):
    #     data= scaled_center_EEG[i, :, j]
    #     plot_eeg_topomap(data, info,chLa_index,i,j, savepath)
    # mean_scaled_center_EEG[i] = np.mean(scaled_center_EEG[i], axis = -1, keepdims=True )
    plot_eeg_topomap(center_EEG_energy[i, :], info, chLa_index, i, i, savepath_energy)
    # for j in range(center_EEG.shape[2]):
    #     data= scaled_center_EEG[i, :, j]
    #     plot_eeg_topomap(data, info,chLa_index,i,j, savepath)
# plot_eeg_topomap(np.mean(center_EEG_energy, axis = 0), info, chLa_index, 7, 7, savepath_energy)

In [ ]:
center_EEG_st = np.std(scaled_center_EEG, axis=-1)
# center_EEG_energy = np.zeros((center_EEG.shape[0],center_EEG.shape[1]))
# for i in range(center_EEG.shape[0]):
#     for j in range(center_EEG.shape[1]):
#         center_EEG_energy[i, j] = np.sum(np.square(center_EEG[i, j, :]))
savepath_std =  "C:/同步文件夹/2024-paper/图/center_EEG_pattern_sub7_stdd60/"
if not os.path.exists(savepath_std):
    os.makedirs(savepath_std)
# scaled_center_EEG = np.zeros((len(cluster_center), 22))
# min_val = np.min(center_EEG_energy)
# max_val = np.max(center_EEG_energy)
# scaled_center_EEG = (center_EEG_energy - min_val) / (max_val - min_val)
for i in range(center_EEG_energy.shape[0]):
    # min_val = np.min(center_EEG_energy[i])
    # max_val = np.max(center_EEG_energy[i])
    # scaled_center_EEG[i] = (center_EEG_energy[i] - min_val) / (max_val - min_val)
    # scaled_center_EEG = scaled_center_EEG
    # for j in range(center_EEG.shape[2]):
    #     data= scaled_center_EEG[i, :, j]
    #     plot_eeg_topomap(data, info,chLa_index,i,j, savepath)
    # mean_scaled_center_EEG[i] = np.mean(scaled_center_EEG[i], axis = -1, keepdims=True )
    plot_eeg_topomap(center_EEG_st[i, :], info, chLa_index, i, i, savepath_energy)
    # for j in range(center_EEG.shape[2]):
    #     data= scaled_center_EEG[i, :, j]
    #     plot_eeg_topomap(data, info,chLa_index,i,j, savepath)

In [ ]:
import seaborn
print(all_att['decoder_attn'].shape)
cor = all_att['decoder_attn'].numpy().squeeze()
plt.figure()
seaborn.heatmap(cor[3].T, vmax=1,vmin=-1)
plt.show()

In [ ]:
import numpy as np
from mne import create_info, concatenate_raws, EpochsArray
from mne.channels import make_standard_montage
import matplotlib.pyplot as plt
from mne.io import RawArray
it = 22
data = cor[it].T
ch_names = [f'state{i+1}' for i in range(data.shape[0])]  # 通道名称
# ch_names.append('Task_Related')
ch_types = ['eeg'] * data.shape[0]  # 通道类型
sfreq = 128  # 采样频率
info = create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sfreq)
raw = RawArray(data, info)
raw.plot(n_channels=data.shape[0], scalings='auto', title='states')
plt.show()

# basis_m = torch.transpose(basis, -1, -2)
# basis_m = basis_m[correct]
# x = torch.tensor(correct_Pooled_x)

# xonbasis  = torch.bmm(x, basis_m)
# data = xonbasis[it].T
# data = torch.concatenate([data, scaler[it].unsqueeze(0)],dim = 0)
# ch_names = [f'pattern{i+1}' for i in range(data.shape[0]-1)]  # 通道名称
# ch_names.append('Task_Related')
# ch_types = ['eeg'] * data.shape[0]  # 通道类型
# sfreq = 64  # 采样频率为1000Hz
# info = create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sfreq)
# raw = RawArray(data, info)
# raw.plot(n_channels=data.shape[0], scalings='auto', title='EEG Signal')
# plt.show()